# labquery demo

Natural language interface for LIMS and liquid handler automation.

This notebook walks through the core workflow:
1. Connect to a LIMS and query samples
2. Check inventory
3. Run a protocol and verify volume writeback
4. Measure a well with the plate reader
5. Ask questions in natural language via Claude

**Prerequisites:** labio-all must be running on localhost:5001. Start it with:
```bash
labquery --simulator &
```
Or run the cell below to start it programmatically.

In [ ]:
# Start labio-all in the background (skip if already running)
from labquery.labio_server import start_labio_server

try:
    labio_proc = start_labio_server()
    print(f"labio-all started (pid {labio_proc.pid})")
except RuntimeError as e:
    print(f"Already running or failed: {e}")

## 1. Connect to the LIMS

In [ ]:
from labquery.lims_client import LabioAllClient

lims = LabioAllClient()

# How many samples are in the LIMS?
all_ids = lims.list_sample_ids()
print(f"Total samples: {len(all_ids)}")
print(f"First 10 IDs: {all_ids[:10]}")

In [ ]:
# Look up a specific sample
sample = lims.get_sample(all_ids[0])
print(f"Sample ID:    {sample.sample_id}")
print(f"Material:     {sample.material_type}")
print(f"Volume:       {sample.volume_ul} {sample.volume_unit}")
print(f"Concentration:{sample.concentration} {sample.concentration_unit}")
print(f"Labware:      {sample.labware_vendor} {sample.labware_catalog}")
print(f"Created:      {sample.created}")

## 2. Check inventory by material type

In [ ]:
from labquery.nl_layer import ToolDispatcher
from labquery.plr_runner import PLRRunner
import json

plr = PLRRunner()
dispatcher = ToolDispatcher(lims, plr)

for material in ["CEL", "DNA", "BAC", "PRO"]:
    result = json.loads(dispatcher.dispatch("check_inventory", {
        "sample_type": material, "min_volume_ul": 100
    }))
    print(f"{material}: {result['available_count']} samples, {result['total_volume_ul']:.0f} uL total")

## 3. Run a protocol and verify LIMS writeback

Find a CEL sample, run the CEL/DNA combination protocol, and confirm the LIMS volume was updated.

In [ ]:
# Find a CEL sample
cel_samples = lims.list_samples(sample_type="CEL", min_volume_ul=200)
target = cel_samples[0]
print(f"Target: {target.sample_id} ({target.material_type}, {target.volume_ul} uL)")

original_volume = target.volume_ul

# Run protocol
result = json.loads(dispatcher.dispatch("run_protocol", {
    "protocol_name": "cel/dna",
    "sample_ids": [target.sample_id],
}))
print(f"\nRun ID: {result['run_id']}")
print(f"Status: {result['status']}")
print(f"Volume consumed: {result['volumes_consumed']}")

# Check writeback
updated = lims.get_sample(target.sample_id)
print(f"\nVolume before: {original_volume} uL")
print(f"Volume after:  {updated.volume_ul} uL")
print(f"Difference:    {original_volume - updated.volume_ul} uL")

## 4. Plate reader measurement

Measure a CEL sample. The plate reader returns a midi-chlorian signal value. BAC and PRO samples are blocked before they reach the reader.

In [ ]:
# Measure a CEL sample
cel_result = json.loads(dispatcher.dispatch("measure_well", {
    "sample_ids": [target.sample_id],
    "volumes": [100.0],
}))
print(f"CEL measurement: {cel_result}")

# Try a BAC sample (should be blocked)
bac_samples = lims.list_samples(sample_type="BAC")
if bac_samples:
    bac_result = json.loads(dispatcher.dispatch("measure_well", {
        "sample_ids": [bac_samples[0].sample_id],
        "volumes": [100.0],
    }))
    print(f"\nBAC measurement (blocked): {bac_result}")

## 5. Natural language queries via Claude

This is the core feature. Ask questions in plain English and Claude uses tool calling to query the LIMS, run protocols, and return answers.

Requires `ANTHROPIC_API_KEY` to be set.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from labquery.nl_layer import NLLayer

nl = NLLayer(lims=lims, plr=plr)

In [ ]:
print(nl.query("How many CEL samples do we have with at least 500 uL?"))

In [ ]:
print(nl.query(f"Look up sample {target.sample_id}. What's its current volume?"))

In [ ]:
print(nl.query("Do we have enough DNA samples for a 384-well plate run?"))

## 6. Available protocols

In [ ]:
protocols = json.loads(dispatcher.dispatch("list_protocols", {}))
for p in protocols:
    print(f"{p['name']}")
    print(f"  Aliases: {', '.join(p['aliases'])}")
    print(f"  Volume per sample: {p['volume_per_sample_ul']} uL")
    print(f"  {p['description']}")
    print()

## Try it yourself

Change the query below to ask anything about the LIMS.

In [ ]:
print(nl.query("Give me a random sample and tell me everything about it"))